# preprocessing the zarr.zip file to get the expression matrix

In [93]:
import anndata as ad
import zarr
import numpy as np
import pandas as pd
from scipy.sparse import csc_matrix
import zipfile
import tempfile
import os
import shutil

class read_xenium_5k:
    """
    A class to read and preprocess the Xenium cell_feature_matrix.zarr.zip
    file into an AnnData object.

    The class handles:
    1. Unzipping the Zarr archive into a temporary directory.
    2. Reading the sparse CSC matrix components (data, indices, indptr) from Zarr.
    3. Constructing the feature metadata (.var) and observation metadata (.obs).
    4. Assembling the final AnnData object.
    5. Cleaning up the temporary directory.

    Note: This class follows the specific Zarr structure provided in the prompt.
    """

    def __init__(self, zip_file_path: str):
        """
        Initializes the loader with the path to the zipped Zarr file.
        
        :param zip_file_path: Path to the 'cell_feature_matrix.zarr.zip' file.
        """
        if not os.path.exists(zip_file_path):
            raise FileNotFoundError(f"Input file not found at: {zip_file_path}")
        self.zip_file_path = zip_file_path
        self.temp_dir = None
        self.zarr_path = None
        self.adata = None
    def hex2string(self, cell_ids, dataset_suffix):
        """
        convert the hex to the cell_id string
        """
        hex_to_shifted = {
                "0": "a",
                "1": "b",
                "2": "c",
                "3": "d",
                "4": "e",
                "5": "f",
                "6": "g",
                "7": "h",
                "8": "i",
                "9": "j",
                "a": "k",
                "b": "l",
                "c": "m",
                "d": "n",
                "e": "o",
                "f": "p"
            }
        # print(self.cell_ids[:10])    

        pad_func = lambda x: x.rjust(8, "0")
        #if you want to map by hex
        # cell_id_prefix = [
        #         pad_func("".join(hex_to_shifted[str(i)] for i in hex(x)[2:]))
        #         for x in cell_ids
        #     ]
        #if you want to map by int
        cell_id_prefix = [
                pad_func("".join(hex_to_shifted[str(i)] for i in str(x)))
                for x in cell_ids
            ]

        cell_string = [
            "".join([i, "-", str(j)])
            for i, j in zip(cell_id_prefix, dataset_suffix)
        ]

        return cell_string


    def _read_zarr_store(self, zarr_store_path: str) -> ad.AnnData:
        """
        Reads the Zarr store components and constructs the AnnData object.

        :param zarr_store_path: Path to the root directory of the unzipped Zarr store.
        :return: An AnnData object.
        """
        print(f"Reading Zarr store from: {self.zip_file_path}")

        # Open the root Zarr group
        store = zarr.ZipStore(self.zip_file_path, mode="r")
        zarr_root = zarr.open(store, mode='r')

        # --- 1. Read Metadata from Group Attributes ---
        attrs = zarr_root["cell_features"].attrs
        number_cells = attrs.get('number_cells')
        number_features = attrs.get('number_features')
        feature_keys = attrs.get('feature_keys')
        feature_ids = attrs.get('feature_ids')
        feature_types = attrs.get('feature_types')

        if not all([number_cells, number_features, feature_keys, feature_types]):
            raise ValueError("Missing critical metadata attributes in Zarr store.")

        # --- 2. Construct .var (Features/Genes) ---
        var_df = pd.DataFrame(
            {'feature_id': feature_ids, 'feature_type': feature_types},
            index=pd.Index(feature_keys, name='feature_name')
        )
        print(f"Created .var with {var_df.shape[0]} features.")
        
        # --- 3. Read and Construct the Sparse Matrix (adata.X) ---
        # Data is stored in Compressed Sparse Column (CSC) format
        try:

            data = zarr_root["cell_features"]['data'][:]
            indices = zarr_root["cell_features"]['indices'][:]
            indptr = zarr_root["cell_features"]['indptr'][:]

        except KeyError as e:
            raise KeyError(f"Missing expected Zarr array component: {e}")

        # Create the scipy CSC matrix
        # Shape is (number_cells, number_features)
        X = csc_matrix(
            (data, indices, indptr),
            shape=(number_cells, number_features)
        )
        print(f"Created sparse matrix X with shape {X.shape}.")

        # --- 4. Construct .obs (Cells) ---
        # The /cell_id array is a 2xN array of uint32, representing (prefix, suffix).
        # We need to convert this to unique string IDs for AnnData's obs_names.
        cell_id_array = zarr_root['cell_features/cell_id'][:]
        
        # Convert the 2-column uint32 cell_id array into a list of strings
        # Format: "{prefix}-{suffix}"
        cell_names = self.hex2string(cell_id_array[:, 0], cell_id_array[:, 1])
        
        obs_df = pd.DataFrame(
            index=pd.Index(cell_names, name='cell_id')
        )
        print(f"Created .obs for {obs_df.shape[0]} cells.")

        # --- 5. Assemble the AnnData Object ---
        adata = ad.AnnData(
            X=X,
            obs=obs_df,
            var=var_df,
            dtype=X.dtype
        )
        # import pdb; pdb.set_trace()
        # Store attributes in .uns for reference
        for key, value in attrs.items():
            if key not in ['feature_ids', 'feature_types', 'feature_keys']:
                adata.uns[key] = value

        print("AnnData object successfully assembled.")
        return adata

    def _cleanup(self):
        """Removes the temporary directory and all extracted files."""
        if self.temp_dir and os.path.exists(self.temp_dir):
            print(f"Cleaning up temporary directory: {self.temp_dir}")
            shutil.rmtree(self.temp_dir)

    def __call__(self) -> ad.AnnData:
        """
        The main method to execute the full loading workflow.

        :return: The final AnnData object.
        """
        try:
            
            # 1. Read the Zarr store and construct AnnData
            adata = self._read_zarr_store(self.zip_file_path)
            # import pdb; pdb.set_trace()
            return adata
        finally:
            # 2. Ensure cleanup happens, even if an error occurred during reading
            self._cleanup()



In [94]:
# Using the requested class name: read_xenium_5k
xenium_loader = read_xenium_5k("/scratch/project_465001820/Spatialformer/data/raw/Xenium_Prime_Human_Skin_FFPE_xe_outs/cell_feature_matrix.zarr.zip")

# Call the instance to run the loading workflow
final_adata = xenium_loader()

Reading Zarr store from: /scratch/project_465001820/Spatialformer/data/raw/Xenium_Prime_Human_Skin_FFPE_xe_outs/cell_feature_matrix.zarr.zip
Created .var with 10018 features.
Created sparse matrix X with shape (112551, 10018).
Created .obs for 112551 cells.
AnnData object successfully assembled.


In [95]:
final_adata

AnnData object with n_obs × n_vars = 112551 × 10018
    var: 'feature_id', 'feature_type'
    uns: 'major_version', 'minor_version', 'number_cells', 'number_features'

In [66]:
hex(1437536272)

'0x55af1010'

In [67]:
final_adata.obs.index.nunique()

112551

In [68]:
final_adata.var

,feature_id,feature_type
feature_name,,
A2ML1,ENSG00000166535,gene
AAMP,ENSG00000127837,gene
AAR2,ENSG00000131043,gene
AARSD1,ENSG00000266967,gene
ABAT,ENSG00000183044,gene
...,...,...
DeprecatedCodeword_18378,DeprecatedCodeword_18378,deprecated_codeword
DeprecatedCodeword_18394,DeprecatedCodeword_18394,deprecated_codeword
DeprecatedCodeword_18534,DeprecatedCodeword_18534,deprecated_codeword


In [72]:
final_adata.var.reset_index().rename(columns={'feature_name': 'gene_name', "feature_id": "gene_ids"}).set_index('gene_ids')

,gene_name,feature_type
gene_ids,,
ENSG00000166535,A2ML1,gene
ENSG00000127837,AAMP,gene
ENSG00000131043,AAR2,gene
ENSG00000266967,AARSD1,gene
ENSG00000183044,ABAT,gene
...,...,...
DeprecatedCodeword_18378,DeprecatedCodeword_18378,deprecated_codeword
DeprecatedCodeword_18394,DeprecatedCodeword_18394,deprecated_codeword
DeprecatedCodeword_18534,DeprecatedCodeword_18534,deprecated_codeword


In [73]:
final_adata.var.index.name = None

In [74]:
final_adata.var

,feature_id,feature_type
A2ML1,ENSG00000166535,gene
AAMP,ENSG00000127837,gene
AAR2,ENSG00000131043,gene
AARSD1,ENSG00000266967,gene
ABAT,ENSG00000183044,gene
...,...,...
DeprecatedCodeword_18378,DeprecatedCodeword_18378,deprecated_codeword
DeprecatedCodeword_18394,DeprecatedCodeword_18394,deprecated_codeword
DeprecatedCodeword_18534,DeprecatedCodeword_18534,deprecated_codeword
DeprecatedCodeword_18649,DeprecatedCodeword_18649,deprecated_codeword


In [69]:
final_adata.obs

""
cell_id
000ehejh-1
000jbedf-1
00bfhgab-1
00bgdbfi-1
00bhfedj-1
...
djagidbbbh-1
djagjejiee-1
djagjfjdab-1
